# PR1 — EDA, education–health indicators & clustering (Uva Province)

**Project:** Machine Learning and Spatial Analysis of Demographic Typology and Education–Health Service Adequacy in Uva Province, Sri Lanka  

**This notebook (expanded PR1):** Same workflow you used before — `BASE=/content`, `pr1_outputs`, crosswalk/Excel loaders — plus **Sri Lanka education-stage proxies**, **facility counts**, **distance merge**, **top/bottom GN tables**, **silhouette K-means**, **priority flags**, and exports to `figures_eda/` and `final/`.

**Upload to Colab `BASE` (default `/content`):**
- `gn_crosswalk_final.csv` **or** `population.xlsx`
- `GN_hospital_school_distances_km.csv`
- `GN_facility_counts_clean.csv` (optional but recommended: `Name`, `Edu_Count`, `Health_Count`)

**Paths:** Set `BASE` below to `"/content"` (upload) or `"/content/drive/MyDrive/YourFolder"` after mounting Drive.

**Wording:** distances reflect **accessibility / proximity**; **adequacy** in discussion should combine need (e.g. school-age, elderly) with access and supply where you have it.


## 1. Environment setup

In [ ]:
# Colab: geospatial + Excel + ML + Parquet
!pip -q install geopandas pyogrio openpyxl scikit-learn pyarrow

In [ ]:
import os
import re
import warnings
import difflib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100

try:
    from IPython.display import display
except ImportError:
    display = print

# --- Paths: Colab uploads in /content/; or Drive: BASE = "/content/drive/MyDrive/YOUR_FOLDER" ---
BASE = "/content"

POP_XLSX = os.path.join(BASE, "population.xlsx")
CROSSWALK_CSV = os.path.join(BASE, "gn_crosswalk_final.csv")
DIST_CSV = os.path.join(BASE, "GN_hospital_school_distances_km.csv")
FAC_CSV = os.path.join(BASE, "GN_facility_counts_clean.csv")

OUT_DIR = os.path.join(BASE, "pr1_outputs")
FIG_DIR = os.path.join(BASE, "figures_eda")
FINAL_DIR = os.path.join(BASE, "final")

for d in (OUT_DIR, FIG_DIR, FINAL_DIR):
    os.makedirs(d, exist_ok=True)

dist = None  # set in distance section
print("OUT_DIR:", OUT_DIR)
print("FIG_DIR:", FIG_DIR)
print("FINAL_DIR:", FINAL_DIR)


## 2. Load data

**Preferred:** `gn_crosswalk_final.csv` (Uva GNs + `district` + age bands + `Name` / `area_name`).

**Otherwise:** `population.xlsx` (sheet `Table 1`) — Uva block between Badulla and Ratnapura rows.

**Facility counts:** merged later on harmonized GN name (case-insensitive).


In [ ]:
def load_uva_from_crosswalk(path: str) -> pd.DataFrame:
    # Load pre-built Uva GN table from project crosswalk CSV.
    df = pd.read_csv(path)
    need = {"district", "gn_number", "total", "0_4", "5_9", "10_14"}
    if not need.issubset(set(df.columns)):
        raise ValueError(f"Crosswalk CSV missing columns. Have: {list(df.columns)[:20]}...")
    return df.copy()


def load_uva_from_excel(path: str) -> pd.DataFrame:
    # Parse Census 2012 hierarchical Excel; keep GN rows for Uva (Badulla + Monaragala).
    raw = pd.read_excel(path, sheet_name="Table 1", header=2)
    raw.columns = [
        "area_name", "gn_number", "total",
        "0_4", "5_9", "10_14", "15_19", "20_24", "25_29", "30_34", "35_39",
        "40_44", "45_49", "50_54", "55_59", "60_64", "65_69", "70_74", "75_79",
        "80_84", "85_89", "90_94", "95_plus",
    ]
    df = raw[raw["area_name"].notna()].copy()
    for c in df.columns[2:]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df[df["area_name"].astype(str).str.lower() != "sri lnka"].copy()
    badulla_start = df.index[df["area_name"].eq("Badulla")][0]
    ratnapura_after = df.index[df["area_name"].eq("Ratnapura")][0]
    uva_block = df.loc[badulla_start : ratnapura_after - 1].copy()
    mona_row = uva_block.index[uva_block["area_name"].eq("Moneragala")][0]
    gn = uva_block[uva_block["gn_number"].notna()].copy()
    gn["district"] = np.where(gn.index < mona_row, "Badulla", "Monaragala")
    gn["gn_number"] = gn["gn_number"].astype(str).str.strip()
    gn.rename(columns={"area_name": "gn_name_pop"}, inplace=True)
    return gn.reset_index(drop=True)


if os.path.isfile(CROSSWALK_CSV):
    df = load_uva_from_crosswalk(CROSSWALK_CSV)
    print("Loaded from crosswalk CSV:", df.shape)
elif os.path.isfile(POP_XLSX):
    df = load_uva_from_excel(POP_XLSX)
    print("Loaded from population.xlsx:", df.shape)
else:
    raise FileNotFoundError("Upload gn_crosswalk_final.csv and/or population.xlsx to BASE folder")

# Harmonize name column for merges (Excel path uses gn_name_pop only)
if "area_name" not in df.columns and "gn_name_pop" in df.columns:
    df["area_name"] = df["gn_name_pop"]
if "Name" not in df.columns and "area_name" in df.columns:
    df["Name"] = df["area_name"]

display(df.head(3))
print("Districts:", df["district"].value_counts().to_dict())


## 3. Data cleaning

In [ ]:
id_col = "gn_number"
df[id_col] = df[id_col].astype(str).str.strip()
dup_n = df.duplicated(id_col).sum()
if dup_n:
    print(f"Warning: {dup_n} duplicate gn_number — keeping first.")
    df = df.drop_duplicates(subset=[id_col], keep="first")

age_cols = [
    "0_4", "5_9", "10_14", "15_19", "20_24", "25_29", "30_34", "35_39",
    "40_44", "45_49", "50_54", "55_59", "60_64", "65_69", "70_74", "75_79",
    "80_84", "85_89", "90_94", "95_plus",
]
for c in age_cols + ["total"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

before = len(df)
df = df[df["total"].notna() & (df["total"] > 0)].copy()
print(f"Rows after valid total filter: {len(df)} (dropped {before - len(df)})")

children = df[["0_4", "5_9", "10_14"]].sum(axis=1)
working = df[["15_19", "20_24", "25_29", "30_34", "35_39", "40_44", "45_49", "50_54", "55_59"]].sum(axis=1)
elderly60 = df[["60_64", "65_69", "70_74", "75_79", "80_84", "85_89", "90_94", "95_plus"]].sum(axis=1)
chk = (children + working + elderly60 - df["total"]).abs()
print("Rows with age sum != total:", int((chk > 0.5).sum()), "| max diff:", float(chk.max()))


## 4. Feature engineering

### 4a Classic structure (as in your original PR1)
Children 0–14, working 15–59, elderly 60+, dependency ratio (young+old 60+) / working.

### 4b Education stages (Census proxies — limitations in comments)
| Stage | Bands | Note |
|-------|-------|------|
| Preschool (ECCD proxy) | `0_4` | includes 0–2 |
| Primary | `5_9` | aligned |
| Junior secondary proxy | `10_14` | age 14 partly senior sec. |
| Senior secondary + collegiate | `15_19` | O/L vs A/L not split |

**Education dependency:** `total_school_age / working_age_15_59 * 100` (working = 15_19…55_59).

### 4c Health-focused elderly
**`p_elderly`** = share aged **65+** (for hospital access). **`p_elderly_60plus`** keeps your original 60+ share for comparison.


In [ ]:
# --- 4a Original PR1 variables ---
df["children"] = df[["0_4", "5_9", "10_14"]].sum(axis=1)
df["working"] = df[["15_19", "20_24", "25_29", "30_34", "35_39", "40_44", "45_49", "50_54", "55_59"]].sum(axis=1)
df["elderly"] = df[["60_64", "65_69", "70_74", "75_79", "80_84", "85_89", "90_94", "95_plus"]].sum(axis=1)

df["p_children"] = (df["children"] / df["total"]) * 100
df["p_working"] = (df["working"] / df["total"]) * 100
df["p_elderly_60plus"] = (df["elderly"] / df["total"]) * 100
df["dependency_ratio"] = ((df["children"] + df["elderly"]) / df["working"].replace(0, np.nan)) * 100

# --- 4b Education-stage proxies (Sri Lanka) ---
df["preschool"] = df["0_4"]
df["primary"] = df["5_9"]
df["junior_secondary"] = df["10_14"]
df["senior_secondary_plus"] = df["15_19"]
df["total_school_age"] = df["preschool"] + df["primary"] + df["junior_secondary"] + df["senior_secondary_plus"]

work_15_59 = df[["15_19", "20_24", "25_29", "30_34", "35_39", "40_44", "45_49", "50_54", "55_59"]].sum(axis=1)
df["working_age_15_59"] = work_15_59
t = df["total"].replace(0, np.nan)
df["p_preschool"] = (df["preschool"] / t) * 100
df["p_primary"] = (df["primary"] / t) * 100
df["p_junior_sec"] = (df["junior_secondary"] / t) * 100
df["p_senior_plus"] = (df["senior_secondary_plus"] / t) * 100
df["p_school_age"] = (df["total_school_age"] / t) * 100
df["education_dependency_ratio"] = (df["total_school_age"] / work_15_59.replace(0, np.nan)) * 100

# --- 4c Elderly 65+ (health focus); p_elderly = 65+ per thesis spec ---
elder_bands_65 = ["65_69", "70_74", "75_79", "80_84", "85_89", "90_94", "95_plus"]
df["elderly_65plus"] = df[elder_bands_65].sum(axis=1)
df["p_elderly"] = (df["elderly_65plus"] / t) * 100

feat_cols = [
    "total", "children", "working", "elderly", "p_children", "p_working", "p_elderly_60plus",
    "dependency_ratio", "p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "p_school_age",
    "education_dependency_ratio", "p_elderly", "elderly_65plus",
]
feat_cols = [c for c in feat_cols if c in df.columns]
df[feat_cols].describe().T.to_csv(os.path.join(OUT_DIR, "01_feature_summary.csv"))
display(df[feat_cols].describe().T.round(2))


## 5. Optional: facility counts (`GN_facility_counts_clean.csv`)

In [ ]:
def norm_join_name(s):
    if pd.isna(s):
        return ""
    return re.sub(r"\s+", " ", str(s).strip().lower())


if os.path.isfile(FAC_CSV):
    fac = pd.read_csv(FAC_CSV)
    fac["name_key_fac"] = fac["Name"].map(norm_join_name)
    agg = fac.groupby("name_key_fac", as_index=False).agg(
        Edu_Count=("Edu_Count", "sum"),
        Health_Count=("Health_Count", "sum"),
    )
    # Match on Name or area_name
    src = df["Name"] if "Name" in df.columns else df["area_name"]
    df["name_key_fac"] = src.map(norm_join_name)
    df = df.merge(agg, left_on="name_key_fac", right_on="name_key_fac", how="left", suffixes=("", "_drop"))
    df.drop(columns=[c for c in df.columns if c.endswith("_drop")], errors="ignore", inplace=True)
    df["Edu_Count"] = df["Edu_Count"].fillna(0)
    df["Health_Count"] = df["Health_Count"].fillna(0)
    df.drop(columns=["name_key_fac"], errors="ignore", inplace=True)
    print("Merged facility counts. Non-zero Edu_Count rows:", int((df["Edu_Count"] > 0).sum()))
else:
    df["Edu_Count"] = 0
    df["Health_Count"] = 0
    print("No facility CSV — Edu_Count/Health_Count set to 0.")


## 6. Data quality check

In [ ]:
key_num = [
    "total", "p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "p_school_age",
    "education_dependency_ratio", "p_elderly", "dependency_ratio",
    "Edu_Count", "Health_Count",
]
key_num = [c for c in key_num if c in df.columns]

print("Shape:", df.shape)
print("\nMissing % (top 12):")
print((df.isna().mean() * 100).sort_values(ascending=False).head(12).round(2).to_string())


def iqr_n(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) < 10:
        return 0
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if iqr <= 0:
        return 0
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum())


print("\nIQR outlier counts:")
for c in key_num:
    print(f"  {c}: {iqr_n(df[c])}")


## 7. Descriptive statistics (full numeric export)

In [ ]:
num = df.select_dtypes(include=[np.number])
desc = num.describe().T
desc.to_csv(os.path.join(OUT_DIR, "02_descriptive_statistics.csv"))
display(desc.round(2).head(25))


## 8. Histograms + KDE (education, health, structure)

In [ ]:
hist_vars = [
    ("total", "Total population per GN"),
    ("p_children", "Share children 0–14 (%)"),
    ("p_elderly_60plus", "Share elderly 60+ (%)"),
    ("p_elderly", "Share elderly 65+ (%) — health focus"),
    ("dependency_ratio", "Broad dependency ratio"),
    ("p_preschool", "Preschool proxy % (0_4)"),
    ("p_primary", "Primary % (5_9)"),
    ("p_junior_sec", "Junior secondary proxy % (10_14)"),
    ("p_senior_plus", "Senior secondary+ % (15_19)"),
    ("p_school_age", "Total school-age % (0_19 proxy)"),
    ("education_dependency_ratio", "Education dependency ratio"),
]

for col, title in hist_vars:
    if col not in df.columns:
        continue
    plt.figure(figsize=(9, 4))
    sns.histplot(df[col].dropna(), kde=True, bins=35, color="#2c7fb8")
    plt.title(title)
    plt.xlabel(col)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"hist_{col}.png"), dpi=200)
    plt.show()
    plt.close()


## 9. Boxplots & violin plots by district

In [ ]:
box_vars = [
    "total", "p_children", "p_elderly", "p_elderly_60plus", "dependency_ratio",
    "p_school_age", "education_dependency_ratio", "p_preschool", "p_primary",
]
box_vars = [c for c in box_vars if c in df.columns]

for col in box_vars:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x="district", y=col, palette="Set2")
    plt.title(f"{col} by district")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"box_{col}_by_district.png"), dpi=200)
    plt.show()
    plt.close()

    plt.figure(figsize=(8, 4))
    sns.violinplot(data=df, x="district", y=col, palette="Pastel1", cut=0, inner="quartile")
    plt.title(f"{col} by district (violin)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"violin_{col}_by_district.png"), dpi=200)
    plt.show()
    plt.close()


## 10. Distances — QGIS export (`GN_hospital_school_distances_km.csv`)

Same as your original PR1 section 13: `GN_code` is a **row index**, not Census `gn_number`. Joining to `df` uses **`name_key`** from `area_name` (median km if several polygons share a name).


In [ ]:
def clean_dist_name(s):
    if pd.isna(s):
        return ""
    t = str(s).replace("\xa0", " ").strip()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"(\d+)$", "", t).strip()
    return t.lower()


def name_key(s):
    s = clean_dist_name(s)
    return re.sub(r"[^a-z0-9]", "", s)


dist = None
if os.path.isfile(DIST_CSV):
    dist = pd.read_csv(DIST_CSV)
    dist.columns = [c.strip() for c in dist.columns]
    dist["Name"] = dist["Name"].astype(str).str.strip()
    dist["name_base"] = dist["Name"].map(clean_dist_name)
    dist["name_key"] = dist["Name"].map(name_key)
    dist["dup_name"] = dist.duplicated("name_key", keep=False)

    for c in ["dist_to_nearest_hospital_km", "dist_to_nearest_school_km"]:
        if c in dist.columns:
            dist[c] = pd.to_numeric(dist[c], errors="coerce")

    print("Distance table:", dist.shape, "| duplicate name_key rows:", int(dist["dup_name"].sum()))
    display(dist[["GN_code", "Name", "dist_to_nearest_hospital_km", "dist_to_nearest_school_km"]].describe())

    for col, title in [
        ("dist_to_nearest_hospital_km", "Distance to nearest hospital (km)"),
        ("dist_to_nearest_school_km", "Distance to nearest school (km)"),
    ]:
        if col not in dist.columns:
            continue
        plt.figure(figsize=(9, 4))
        sns.histplot(dist[col].dropna(), kde=True, bins=40, color="#3182bd")
        plt.title(title)
        plt.xlabel("km")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, f"hist_{col}.png"), dpi=200)
        plt.show()
        plt.close()

    if "dist_to_nearest_school_km" in dist.columns and "dist_to_nearest_hospital_km" in dist.columns:
        plt.figure(figsize=(6, 6))
        plt.scatter(dist["dist_to_nearest_school_km"], dist["dist_to_nearest_hospital_km"], alpha=0.35, s=22)
        plt.xlabel("km to nearest school")
        plt.ylabel("km to nearest hospital")
        plt.title("School vs hospital distance (GN polygons)")
        mx = dist["dist_to_nearest_school_km"].max()
        plt.plot([0, mx], [0, mx], "k--", alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "scatter_school_vs_hospital_km.png"), dpi=200)
        plt.show()
        plt.close()

    for col in ["dist_to_nearest_hospital_km", "dist_to_nearest_school_km"]:
        if col not in dist.columns:
            continue
        med, mx = dist[col].median(), dist[col].max()
        if med and med > 0:
            print(f"{col}: median={med:.2f} km, max={mx:.2f} km, max/median={mx/med:.2f}")
else:
    print("Skip distances: upload GN_hospital_school_distances_km.csv to BASE")


### 11. Merge distances into `df` (median per `name_key`)

Creates `dist_hospital_km`, `dist_school_km`, `n_polygons`. Optional **fuzzy** rename of `name_key` when Census and QGIS names differ slightly.


In [ ]:
if dist is not None and "area_name" in df.columns:
    dist_agg = (
        dist.groupby("name_key", as_index=False)
        .agg(
            n_polygons=("GN_code", "count"),
            dist_hospital_km=("dist_to_nearest_hospital_km", "median"),
            dist_school_km=("dist_to_nearest_school_km", "median"),
            dist_hospital_max=("dist_to_nearest_hospital_km", "max"),
            dist_school_max=("dist_to_nearest_school_km", "max"),
        )
    )
    dist_agg = dist_agg[dist_agg["name_key"].notna()].copy()
    valid_keys = set(dist_agg["name_key"].astype(str))
    df["name_key"] = df["area_name"].map(name_key)
    pool = sorted(k for k in valid_keys if k and str(k).lower() != "nan")

    for i in df.index:
        nk = str(df.at[i, "name_key"]) if pd.notna(df.at[i, "name_key"]) else ""
        if nk and nk not in valid_keys:
            m = difflib.get_close_matches(nk, pool, n=1, cutoff=0.82)
            if m:
                df.at[i, "name_key"] = m[0]

    drop_m = [c for c in ["dist_hospital_km", "dist_school_km", "n_polygons", "dist_hospital_max", "dist_school_max"] if c in df.columns]
    df = df.drop(columns=drop_m, errors="ignore")
    df = df.merge(dist_agg, on="name_key", how="left")
    matched = df["dist_hospital_km"].notna().sum()
    print(f"Census GN rows with distance match: {matched} / {len(df)}")

    sub = df[df["dist_hospital_km"].notna()].copy()
    if len(sub) > 10:
        for col, lab in [
            ("dist_hospital_km", "Median km to nearest hospital"),
            ("dist_school_km", "Median km to nearest school"),
        ]:
            plt.figure(figsize=(8, 4))
            sns.boxplot(data=sub, x="district", y=col, palette="Set2")
            plt.title(lab)
            plt.tight_layout()
            plt.savefig(os.path.join(FIG_DIR, f"box_{col}_by_district.png"), dpi=200)
            plt.show()
            plt.close()

    ok = [c for c in ["dist_hospital_km", "dist_school_km", "p_children", "p_elderly", "p_school_age", "dependency_ratio", "total"] if c in df.columns]
    if len(ok) >= 2:
        cc = df[ok].dropna().corr()
        plt.figure(figsize=(8, 6))
        sns.heatmap(cc, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
        plt.title("Distance vs demographics (matched rows)")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "heatmap_dist_demographics.png"), dpi=200)
        plt.show()
        plt.close()
        cc.to_csv(os.path.join(OUT_DIR, "07_correlation_dist_demographics.csv"))

    df.to_csv(os.path.join(OUT_DIR, "uva_gn_pr1_with_distances.csv"), index=False)
    print("Saved:", os.path.join(OUT_DIR, "uva_gn_pr1_with_distances.csv"))
else:
    print("Skip merge: need distance file and area_name in df.")


## 12. Top / bottom GN tables (for slides)

In [ ]:
label_cols = ["gn_number", "district"]
if "Name" in df.columns:
    label_cols.insert(1, "Name")
elif "area_name" in df.columns:
    label_cols.insert(1, "area_name")
label_cols = [c for c in label_cols if c in df.columns]

rank_vars = [
    "p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "p_school_age",
    "education_dependency_ratio", "p_elderly", "dist_school_km", "dist_hospital_km",
]
rank_vars = [c for c in rank_vars if c in df.columns]

y_bar = "Name" if "Name" in label_cols else ("area_name" if "area_name" in label_cols else label_cols[-1])

for v in rank_vars:
    sub = df[label_cols + [v]].dropna(subset=[v]).sort_values(v, ascending=False)
    print(f"\n=== Top 10 — {v} ===")
    display(sub.head(10).reset_index(drop=True))
    print(f"=== Bottom 10 — {v} ===")
    display(sub.tail(10).sort_values(v).reset_index(drop=True))
    plt.figure(figsize=(8, 4))
    sns.barplot(data=sub.head(10), y=y_bar, x=v, orient="h", color="coral")
    plt.title(f"Top 10 GNs — {v}")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"bar_top10_{v}.png"), dpi=200)
    plt.show()
    plt.close()

for v in ["dist_school_km", "dist_hospital_km"]:
    if v not in df.columns:
        continue
    sub = df[label_cols + [v]].dropna(subset=[v]).nsmallest(10, v)
    print(f"\n=== Best served (smallest {v}) — 10 closest ===")
    display(sub.reset_index(drop=True))
    plt.figure(figsize=(8, 4))
    sns.barplot(data=sub, y=y_bar, x=v, orient="h", color="teal")
    plt.title(f"Closest 10 GNs — {v}")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"bar_closest10_{v}.png"), dpi=200)
    plt.show()
    plt.close()


## 13. Correlation heatmap (demographics + education + facilities + distances)

In [ ]:
corr_cols = [
    "total", "children", "working", "elderly", "p_children", "p_working", "p_elderly_60plus",
    "dependency_ratio", "p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "p_school_age",
    "education_dependency_ratio", "p_elderly", "Edu_Count", "Health_Count",
    "dist_school_km", "dist_hospital_km",
]
corr_cols = [c for c in corr_cols if c in df.columns]
corr = df[corr_cols].corr()
corr.to_csv(os.path.join(OUT_DIR, "03_correlation_matrix.csv"))
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=False, fmt=".2f", cmap="RdBu_r", center=0, square=False)
plt.title("Correlation — Uva GN (demographics + education + facilities)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "correlation_heatmap_full.png"), dpi=200)
plt.show()
plt.close()


## 14. Scatter plots (relationships)

In [ ]:
def sc(x, y, fname):
    if x not in df.columns or y not in df.columns:
        return
    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=df, x=x, y=y, hue="district", alpha=0.65, s=40)
    plt.title(f"{y} vs {x}")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=200)
    plt.show()
    plt.close()


sc("p_school_age", "dist_school_km", "scatter_p_school_age_vs_dist_school.png")
sc("p_elderly", "dist_hospital_km", "scatter_p_elderly_vs_dist_hospital.png")
sc("education_dependency_ratio", "total", "scatter_edudep_vs_total.png")
sc("total", "dist_school_km", "scatter_total_vs_dist_school.png")
print("(Scatter plots use dist_* columns after section 11 merge.)")


## 15. Pairplot — education-stage percentages

In [ ]:
pair_cols = [c for c in ["p_preschool", "p_primary", "p_junior_sec", "p_senior_plus"] if c in df.columns]
if len(pair_cols) >= 2:
    g = sns.pairplot(df[pair_cols].dropna(), corner=True, plot_kws={"s": 25, "alpha": 0.5})
    g.fig.suptitle("Education-stage shares (%) — pairplot", y=1.02)
    plt.savefig(os.path.join(FIG_DIR, "pairplot_education_stages.png"), dpi=200)
    plt.show()
    plt.close("all")
else:
    print("Skip pairplot — need stage percentage columns.")


## 16. District comparison (Badulla vs Monaragala)

In [ ]:
demo_cols = ["total", "children", "working", "elderly", "p_children", "p_working", "p_elderly_60plus", "dependency_ratio"]
demo_cols = [c for c in demo_cols if c in df.columns]
summary = df.groupby("district")[demo_cols].agg(["mean", "median", "std"])
summary.to_csv(os.path.join(OUT_DIR, "04_district_comparison.csv"))
display(summary.round(2))

melted = df.melt(
    id_vars=["district"],
    value_vars=[c for c in ["p_children", "p_elderly", "p_school_age", "dependency_ratio"] if c in df.columns],
    var_name="metric",
    value_name="value",
)
if len(melted):
    plt.figure(figsize=(11, 5))
    sns.barplot(data=melted, x="metric", y="value", hue="district", errorbar="sd", palette="muted")
    plt.title("Mean indicators by district (error bars = SD)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "district_comparison_bars.png"), dpi=200)
    plt.show()
    plt.close()


## 17. K-means — education typology (silhouette + PCA)

**Features:** `p_preschool`, `p_primary`, `p_junior_sec`, `p_senior_plus`, `log_total`, `dependency_ratio`.  

Standardize → silhouette for **k = 2…10** → choose best **k** → cluster labels + PCA plot (as in your original PR1).


In [ ]:
clust_feats = [c for c in ["p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "dependency_ratio"] if c in df.columns]
if "total" not in df.columns:
    raise ValueError("Need total for log_total")

Xdf = df[clust_feats + ["total"]].replace([np.inf, -np.inf], np.nan)
Xdf["log_total"] = np.log1p(Xdf["total"])
Xdf = Xdf.drop(columns=["total"])
Xdf = Xdf.fillna(Xdf.median(numeric_only=True))

scaler_k = StandardScaler()
Xz_k = scaler_k.fit_transform(Xdf)

sil_scores = {}
best_k, best_s = None, -1.0
for k in range(2, 11):
    if len(Xz_k) <= k:
        continue
    try:
        km_try = KMeans(n_clusters=k, random_state=42, n_init="auto")
    except TypeError:
        km_try = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km_try.fit_predict(Xz_k)
    try:
        s = silhouette_score(Xz_k, lab)
        sil_scores[k] = s
        if s > best_s:
            best_k, best_s = k, s
    except Exception:
        pass

if best_k is None:
    best_k = 4

print("Silhouette (k=2..10):", {k: round(v, 4) for k, v in sorted(sil_scores.items())})
print("Chosen k:", best_k, "| best silhouette:", round(best_s, 4))

k = best_k
try:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
except TypeError:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster_kmeans"] = km.fit_predict(Xz_k)

prof_cols = [c for c in ["p_preschool", "p_primary", "p_junior_sec", "p_senior_plus", "total"] if c in df.columns]
cluster_summary = df.groupby("cluster_kmeans")[prof_cols].agg(["mean", "median", "count"])
cluster_summary.to_csv(os.path.join(OUT_DIR, "05_kmeans_cluster_summary.csv"))
display(cluster_summary.round(2))
print(df["cluster_kmeans"].value_counts().sort_index())

plt.figure(figsize=(max(6, k + 2), 4))
sns.heatmap(df.groupby("cluster_kmeans")[prof_cols].mean().T, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Cluster profiles — mean education % and total")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "kmeans_cluster_means_heatmap.png"), dpi=200)
plt.show()
plt.close()


## 18. PCA scatter (clusters) — same style as your PR1

In [ ]:
pca = PCA(n_components=2, random_state=42)
xy = pca.fit_transform(Xz_k)
plot_df = pd.DataFrame(
    {"PC1": xy[:, 0], "PC2": xy[:, 1], "cluster": df["cluster_kmeans"].astype(str), "district": df["district"]}
)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="cluster", style="district", palette="deep", s=55, alpha=0.85)
plt.title(f"K-means clusters (k={k}) — PCA projection")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "kmeans_pca_scatter.png"), dpi=200)
plt.show()
plt.close()

load_cols = list(Xdf.columns)
loadings = pd.DataFrame(pca.components_.T, index=load_cols, columns=["PC1", "PC2"])
loadings.to_csv(os.path.join(OUT_DIR, "06_pca_loadings.csv"))
display(loadings.round(3))


## 19. Priority GN flags (education + health hotspots)

Uses **median km** from section 11 (matched rows only). Percentiles are computed on rows with non-missing distances.

- `priority_education`: `p_school_age` and `dist_school_km` both ≥ **75th percentile**.
- `priority_health`: `p_elderly` (65+) and `dist_hospital_km` both ≥ **75th percentile**.
- `priority_dual`: either flag.


In [ ]:
if "dist_school_km" in df.columns and df["dist_school_km"].notna().any():
    m = df["dist_school_km"].notna() & df["dist_hospital_km"].notna()
    sub = df.loc[m].copy()
    df["priority_education"] = False
    df["priority_health"] = False
    th_sch = sub["p_school_age"].quantile(0.75) if "p_school_age" in sub.columns else np.nan
    th_dist_s = sub["dist_school_km"].quantile(0.75)
    th_el = sub["p_elderly"].quantile(0.75)
    th_dist_h = sub["dist_hospital_km"].quantile(0.75)
    df.loc[m, "priority_education"] = (sub["p_school_age"] >= th_sch) & (sub["dist_school_km"] >= th_dist_s)
    df.loc[m, "priority_health"] = (sub["p_elderly"] >= th_el) & (sub["dist_hospital_km"] >= th_dist_h)
    df["priority_dual"] = df["priority_education"] | df["priority_health"]
    flagged = df[df["priority_dual"]]
    print("Flagged GNs:", len(flagged))
    show_c = [c for c in ["gn_number", "Name", "area_name", "district", "p_school_age", "dist_school_km", "p_elderly", "dist_hospital_km"] if c in df.columns]
    display(flagged[show_c].head(30))
else:
    df["priority_education"] = False
    df["priority_health"] = False
    df["priority_dual"] = False
    print("No distance merge — upload CSV and run sections 10–11, then re-run this cell.")


## 20. Final exports (`final/` folder)

In [ ]:
out_master = os.path.join(FINAL_DIR, "UVA_master_dataset_education_health.csv")
df.to_csv(out_master, index=False)
print("Saved:", out_master)

try:
    df.to_parquet(os.path.join(FINAL_DIR, "UVA_master_dataset_education_health.parquet"), index=False)
    print("Saved Parquet OK")
except Exception as e:
    print("Parquet skipped:", e)

df.to_csv(os.path.join(OUT_DIR, "uva_gn_pr1_cleaned_with_clusters.csv"), index=False)
print("Also saved PR1 copy:", os.path.join(OUT_DIR, "uva_gn_pr1_cleaned_with_clusters.csv"))


## 21. Auto summary + presentation bullets

In [ ]:
print("--- Run summary ---")
print(f"GN divisions: {len(df)}")
print(f"Mean p_school_age (%): {df['p_school_age'].mean():.2f}" if "p_school_age" in df.columns else "")
print(f"Mean p_elderly 65+ (%): {df['p_elderly'].mean():.2f}" if "p_elderly" in df.columns else "")
print(f"Mean dependency_ratio: {df['dependency_ratio'].mean():.2f}")
if "dist_school_km" in df.columns:
    print(f"Matched distances: {df['dist_school_km'].notna().sum()} / {len(df)}")
if "priority_dual" in df.columns:
    print(f"Priority dual flags: {int(df['priority_dual'].sum())}")

bullets = [
    f"Data: {len(df)} GNs with age structure; districts Badulla vs Monaragala.",
    "Education proxies: preschool (0_4), primary (5_9), junior sec. proxy (10_14), senior+ (15_19); see markdown for limitations.",
    "K-means uses stage shares + log total + broad dependency_ratio; silhouette picks k; PCA plot for slides.",
    "Distances: median km per GN name (polygon aggregation); correlate with p_school_age and p_elderly (65+).",
    "Priority GNs: high need (school-age or elderly share) AND far from nearest school/hospital (75th pct among matched).",
    f"Outputs: CSV/Parquet in `{FINAL_DIR}`, figures in `{FIG_DIR}`, tables in `{OUT_DIR}`.",
    "Maps: join `UVA_master_dataset_education_health.csv` to GN boundaries in QGIS on `Name` / `area_name` / `gn_number` as appropriate.",
]
for b in bullets:
    print("•", b)


## 22. (Optional) GeoPackages / shapefiles on BASE

In [ ]:
# import geopandas as gpd
# b = gpd.read_file(os.path.join(BASE, "Badulla District GN Boundary.shp"))
# print(len(b), b.crs)
